# Code to access CMIP 6 data
Only to keep it, will not be necessary for master thesis anymore.

- CMIP6 historical and projection data for 3 different pathways
- [CMIP derived climate extreme indices](https://cds.climate.copernicus.eu/datasets/projections-cmip6?tab=overview)

On this page I can search for different models and experiments and can generate query code for **climate projections** of heat indicators. Climate data store also has other dataset that indluce projections where I could get other variables from.

In [ ]:
# libraries
import cdsapi
import tempfile
import zipfile
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
def download_and_open_dataset(dataset, request):

    """
    Downloads a dataset from the CDS API and opens it in memory.
    Parameters:
    - dataset (str): The name of the dataset to download.
    - request (dict): The request parameters for the dataset.
    Returns:
    - ds (xr.Dataset): The dataset opened in memory.
    """


    client = cdsapi.Client()

    # Create a temporary file for the ZIP download
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=True) as tmp_zip:
        # Download the ZIP file to the temporary path
        client.retrieve(dataset, request).download(tmp_zip.name)

        # Open the ZIP file from the temporary file
        with zipfile.ZipFile(tmp_zip.name, 'r') as zip_ref:
            # Get list of .nc files
            nc_files = [name for name in zip_ref.namelist() if name.endswith('.nc')]
            if len(nc_files) != 1:
                raise ValueError(f"Expected one .nc file, found: {nc_files}")
            
            # Read .nc file into memory
            nc_data = zip_ref.read(nc_files[0])
            nc_buffer = io.BytesIO(nc_data)

            # Load dataset from memory buffer
            ds = xr.open_dataset(nc_buffer)

    return ds

### Historical data for climate variables

In [ ]:
## Create requests
dataset = 'projections-cmip6'

## Daily data
# Precipitation
request_precip_83007 = {'temporal_resolution': 'daily',
 'experiment': 'historical',
 'variable': 'precipitation',
 'model': 'ec_earth3_cc',
 'month': ['03','04','05','06','07','08','09','10','11','12'], # Two months buffer around event
 'day': ['01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24','25','26','27','28','29','30','31'],
 'year': ['2010']}

# Near surface max air temperature
request_tempmax_83007 = {'temporal_resolution': 'daily',
 'experiment': 'historical',
 'variable': 'daily_maximum_near_surface_air_temperature',
 'model': 'ec_earth3_cc',
 'month': ['03','04','05','06','07','08','09','10','11','12'], # Two months buffer around event 
 'day': ['01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24','25','26','27','28','29','30','31'],
 'year': ['2010']}

# Near surface min air temperature
request_tempmin_83007 = {'temporal_resolution': 'daily',
 'experiment': 'historical',
 'variable': 'daily_minimum_near_surface_air_temperature',
 'model': 'ec_earth3_cc',
 'month': ['03','04','05','06','07','08','09','10','11','12'], # Two months buffer around event 
 'day': ['01','02','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','22','23','24','25','26','27','28','29','30','31'],
 'year': ['2010']}

## Monthly data
# Soil moisture
#TODO: tbc

In [ ]:
ds_precip_83007 = download_and_open_dataset(dataset, request_precip_83007)
ds_tempmax_83007 = download_and_open_dataset(dataset, request_tempmax_83007)
ds_tempmin_83007 = download_and_open_dataset(dataset, request_tempmin_83007)

In [ ]:
# Examine dataset
ds_tempmax_83007

### Have a look at the data

In [ ]:
# Extract the precipitation data from the first time slice
precip_data = ds_tempmax_83007['tasmax'].isel(time=0)

# Create a figure and axis with a PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot the precipitation data
precip_data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis')

# Add coastlines
ax.coastlines()

# Add gridlines
ax.gridlines(draw_labels=True)

# Show the plot
plt.show()

### Projected data, heat and drought indices

In [ ]:
dataset_chd_indicators = "sis-extreme-indices-cmip6"

common_request_params = {
    "product_type": ["bias_adjusted"],
    "model": ["mpi_esm1_2_lr"],
    "ensemble_member": ["r1i1p1f1"],
    "temporal_aggregation": ["daily"],
    "period": ["20110101_21001231"],
    "version": ["2_0"]
}

# Heat index requests
request_heat_index_ssp1_26 = {
    "variable": ["heat_index"],
    "experiment": ["ssp1_2_6"],
    **common_request_params
}

request_heat_index_ssp2_45 = {
    "variable": ["heat_index"],
    "experiment": ["ssp2_4_5"],
    **common_request_params
}

request_heat_index_ssp3_70 = {
    "variable": ["heat_index"],
    "experiment": ["ssp3_7_0"],
    **common_request_params
}

request_heat_index_ssp5_85 = {
    "variable": ["heat_index"],
    "experiment": ["ssp5_8_5"],
    **common_request_params
}

# Humidex requests
request_humidex_ssp1_26 = {
    "variable": ["humidex"],
    "experiment": ["ssp1_2_6"],
    **common_request_params
}

request_humidex_ssp2_45 = {
    "variable": ["humidex"],
    "experiment": ["ssp2_4_5"],
    **common_request_params
}

request_humidex_ssp3_70 = {
    "variable": ["humidex"],
    "experiment": ["ssp3_7_0"],
    **common_request_params
}

request_humidex_ssp5_85 = {
    "variable": ["humidex"],
    "experiment": ["ssp5_8_5"],
    **common_request_params
}


In [ ]:
ds_heat_ssp1_26 = download_and_open_dataset(dataset_chd_indicators, request_heat_index_ssp1_26)

In [ ]:
ds_humid_ssp1_26 = download_and_open_dataset(dataset_chd_indicators, request_humidex_ssp1_26)

In [ ]:
# Examine dataset
ds_heat_ssp1_26

In [ ]:
ds_humid_ssp1_26

#### Compare with plot event
This is one of the middle days of event 83007 in southeast asia. The high heat index around thailand corresponds to the event. 

TODO to check indicator suitability:
- do same with precipitation extreme indicator
- check visually where this is extreme
- make a boolean mask where both heat and precipitation are extreme (over a certain threshold)
- compare with event mask
- check this datasets variables https://cds.climate.copernicus.eu/datasets/sis-biodiversity-cmip5-global?tab=overview

In [ ]:
# Extract the hrat index data for around the middle of the event
heat_index = ds_heat_ssp1_26['HI'].isel(time=1886)

# Create a figure and axis with a PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot the precipitation data
heat_index.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis')

# Add coastlines
ax.coastlines()

# Add gridlines
ax.gridlines(draw_labels=True)

# Show the plot
plt.show()

In [ ]:
# Extract the hrat index data for around the middle of the event
humid_index = ds_humid_ssp1_26['Humidex'].isel(time=1886)

# Create a figure and axis with a PlateCarree projection
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot the precipitation data
humid_index.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis')

# Add coastlines
ax.coastlines()

# Add gridlines
ax.gridlines(draw_labels=True)

# Show the plot
plt.show()